# E-Commerce Customer & Sales Analysis

**Dataset:** UCI Online Retail  
**Project focus:** Data cleaning, exploratory data analysis, visualisation, RFM customer analysis, and K-Means customer segmentation.

### Objectives
- Load and understand the Online Retail dataset
- Clean duplicates, cancellations, invalid quantities, and invalid prices
- Create Revenue, Month, DayOfWeek, and Hour features
- Analyse revenue, orders, customers, products, countries, and monthly trends
- Build RFM (Recency, Frequency, Monetary) customer features
- Segment customers using K-Means clustering
- Produce business insights and recommendations

> **How to run:** In Google Colab choose **Runtime → Run all**.  
> If `Online Retail.xlsx` is not already present, the notebook will download it automatically from the UCI Machine Learning Repository.

## 1. Imports

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)

## 2. Get and Load the Dataset

In [ ]:
DATA_FILE = Path("Online Retail.xlsx")
ZIP_FILE = Path("online_retail.zip")
DATA_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

if not DATA_FILE.exists():
    print("Dataset not found locally. Downloading from UCI...")
    urllib.request.urlretrieve(DATA_URL, ZIP_FILE)

    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        z.extractall(".")

    print("Dataset downloaded and extracted successfully.")
else:
    print("Dataset already available locally.")

print("Dataset file:", DATA_FILE.resolve())

In [ ]:
df = pd.read_excel(DATA_FILE)
print(f"Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 3. Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
df.describe(include='all', datetime_is_numeric=True) if "datetime_is_numeric" in pd.DataFrame.describe.__code__.co_varnames else df.describe(include='all')

### Data Understanding Notes

The dataset contains invoice-level retail transactions. Important fields include invoice number, product description, quantity, invoice date, unit price, customer ID, and country.

Missing `CustomerID` values are retained in the main sales dataset because those transactions can still contribute to overall sales analysis. A separate customer dataset is created later for RFM analysis, where a valid customer identifier is required.

## 4. Data Cleaning

In [ ]:
clean_df = df.copy()
cleaning_summary = []

# 1. Duplicates
duplicate_rows = int(clean_df.duplicated().sum())
clean_df = clean_df.drop_duplicates().copy()
cleaning_summary.append(("Duplicate rows removed", duplicate_rows))

# 2. Cancelled orders (InvoiceNo starting with C)
cancelled_rows = int(clean_df['InvoiceNo'].astype(str).str.startswith('C').sum())
clean_df = clean_df[
    ~clean_df['InvoiceNo'].astype(str).str.startswith('C')
].copy()
cleaning_summary.append(("Cancelled-order rows removed", cancelled_rows))

# 3. Non-positive quantities
invalid_quantity_rows = int((clean_df['Quantity'] <= 0).sum())
clean_df = clean_df[clean_df['Quantity'] > 0].copy()
cleaning_summary.append(("Non-positive quantity rows removed", invalid_quantity_rows))

# 4. Non-positive prices
invalid_price_rows = int((clean_df['UnitPrice'] <= 0).sum())
clean_df = clean_df[clean_df['UnitPrice'] > 0].copy()
cleaning_summary.append(("Non-positive price rows removed", invalid_price_rows))

cleaning_summary_df = pd.DataFrame(
    cleaning_summary,
    columns=["Cleaning Step", "Rows Removed"]
)

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(clean_df):,}")
cleaning_summary_df

In [ ]:
print("Missing values after transaction cleaning:")
clean_df.isnull().sum()

`CustomerID` can be missing for otherwise valid sales transactions. Therefore:

- `clean_df` is used for **overall sales analysis**.
- `customer_df` contains only transactions with a valid `CustomerID` and is used for **customer/RFM analysis**.

## 5. Feature Engineering

In [ ]:
clean_df['Revenue'] = clean_df['Quantity'] * clean_df['UnitPrice']
clean_df['Month'] = clean_df['InvoiceDate'].dt.month
clean_df['DayOfWeek'] = clean_df['InvoiceDate'].dt.day_name()
clean_df['Hour'] = clean_df['InvoiceDate'].dt.hour

clean_df[['InvoiceDate', 'Quantity', 'UnitPrice', 'Revenue', 'Month', 'DayOfWeek', 'Hour']].head()

In [ ]:
customer_df = clean_df.dropna(subset=['CustomerID']).copy()
customer_df['CustomerID'] = customer_df['CustomerID'].astype(int)

print(f"Customer-analysis rows: {len(customer_df):,}")
print(f"Unique customers: {customer_df['CustomerID'].nunique():,}")
customer_df.head()

## 6. Exploratory Data Analysis

In [ ]:
total_revenue = clean_df['Revenue'].sum()
total_orders = clean_df['InvoiceNo'].nunique()
total_customers = customer_df['CustomerID'].nunique()

summary_metrics = pd.DataFrame({
    "Metric": ["Total Revenue (£)", "Total Orders", "Total Customers"],
    "Value": [round(total_revenue, 2), total_orders, total_customers]
})

print(f"Total Revenue: £{total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
summary_metrics

### Top 10 Best-Selling Products

In [ ]:
top_products = (
    clean_df.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products.to_frame("Quantity Sold")

### Top 10 Countries by Revenue

In [ ]:
top_countries = (
    clean_df.groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_countries.to_frame("Revenue (£)")

### Top 10 Customers by Spending

In [ ]:
top_customers = (
    customer_df.groupby('CustomerID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_customers.to_frame("Total Spending (£)")

### Monthly Revenue

In [ ]:
monthly_sales = (
    clean_df.groupby(clean_df['InvoiceDate'].dt.to_period('M'))['Revenue']
    .sum()
)

monthly_sales.to_frame("Revenue (£)")

## 7. Visualisations

In [ ]:
Path("images").mkdir(exist_ok=True)

### Monthly Revenue Trend

In [ ]:
plt.figure(figsize=(12, 6))
monthly_sales.plot(marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.savefig("images/monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** Revenue rises strongly in the later months of 2011, with November showing the highest monthly revenue. The final December figure should be interpreted carefully because the dataset ends on 9 December 2011, so December is incomplete.

### Top 10 Best-Selling Products

In [ ]:
plt.figure(figsize=(12, 6))
top_products.sort_values().plot(kind='barh')
plt.title('Top 10 Best-Selling Products')
plt.xlabel('Quantity Sold')
plt.ylabel('Product')
plt.tight_layout()
plt.savefig("images/top_products.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** The leading products by quantity substantially outperform several other products in the top 10, indicating particularly strong demand for these items.

### Top 10 Countries by Revenue

In [ ]:
plt.figure(figsize=(12, 6))
top_countries.sort_values().plot(kind='barh')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue (£)')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig("images/top_countries.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** The United Kingdom is the dominant revenue market. The strongest international markets include the Netherlands, EIRE, Germany, and France.

### Top 10 Customers by Spending

In [ ]:
plt.figure(figsize=(12, 6))
top_customers.sort_values().plot(kind='barh')
plt.title('Top 10 Customers by Spending')
plt.xlabel('Total Spending (£)')
plt.ylabel('Customer ID')
plt.tight_layout()
plt.savefig("images/top_customers.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. RFM Customer Analysis

RFM means:

- **Recency** — number of days since the customer's most recent purchase. Lower is better.
- **Frequency** — number of unique orders made by the customer. Higher is better.
- **Monetary** — total revenue generated by the customer. Higher is better.

In [ ]:
reference_date = customer_df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = customer_df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

print("Reference date:", reference_date)
rfm.head()

In [ ]:
rfm.describe()

## 9. Prepare RFM Data for K-Means

RFM features have very different ranges and are highly skewed. We first apply a log transformation to reduce the effect of extreme values, then standardise the features so that Recency, Frequency, and Monetary contribute on a comparable scale.

In [ ]:
rfm_features = rfm[['Recency', 'Frequency', 'Monetary']].copy()

rfm_log = np.log1p(rfm_features)

scaler = StandardScaler()
rfm_log_scaled = scaler.fit_transform(rfm_log)

rfm_log.head()

### Elbow Method

In [ ]:
k_values = list(range(2, 11))
inertia_log = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(rfm_log_scaled)
    inertia_log.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia_log, marker='o')
plt.title('Elbow Method After Log Transformation')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(k_values)
plt.grid(True)
plt.tight_layout()
plt.savefig("images/elbow_method.png", dpi=150, bbox_inches="tight")
plt.show()

### Silhouette Score

In [ ]:
silhouette_scores = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = model.fit_predict(rfm_log_scaled)
    score = silhouette_score(rfm_log_scaled, labels)
    silhouette_scores.append(score)
    print(f"K = {k}: {score:.3f}")

best_k_by_silhouette = k_values[int(np.argmax(silhouette_scores))]
print(f"\nHighest silhouette score occurs at K = {best_k_by_silhouette}.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_scores, marker='o')
plt.title('Silhouette Scores by Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(k_values)
plt.grid(True)
plt.tight_layout()
plt.savefig("images/silhouette_scores.png", dpi=150, bbox_inches="tight")
plt.show()

### Why use 4 clusters?

The silhouette score may favour a smaller number of clusters, while the elbow curve shows diminishing gains as the number of clusters increases. For this project, **K = 4** is selected as a practical balance between statistical separation and business interpretability.

Four groups allow the customer base to be described in actionable terms such as high-value loyal customers, inactive/at-risk customers, promising recent customers, and regular customers. This choice is explicitly documented rather than treating the statistical metric as the only decision criterion.

## 10. Final K-Means Customer Segmentation

In [ ]:
FINAL_K = 4

final_kmeans = KMeans(
    n_clusters=FINAL_K,
    random_state=42,
    n_init=10
)

rfm['Cluster'] = final_kmeans.fit_predict(rfm_log_scaled)

rfm['Cluster'].value_counts().sort_index()

In [ ]:
cluster_profile = (
    rfm.groupby('Cluster')
    .agg(
        Recency=('Recency', 'mean'),
        Frequency=('Frequency', 'mean'),
        Monetary=('Monetary', 'mean'),
        Customers=('Cluster', 'size')
    )
    .round(2)
)

cluster_profile

### Assign Business-Friendly Segment Names

In [ ]:
# Assign names from the actual cluster profile so the notebook is robust
# even if numeric cluster labels change between environments.

remaining = set(cluster_profile.index)

# High-Value Loyal: strongest combination of high frequency, high monetary,
# and low recency.
value_score = (
    cluster_profile['Frequency'].rank(pct=True)
    + cluster_profile['Monetary'].rank(pct=True)
    - cluster_profile['Recency'].rank(pct=True)
)
high_value_cluster = value_score.idxmax()
remaining.remove(high_value_cluster)

# At-Risk / Inactive: highest recency among the remaining clusters.
at_risk_cluster = cluster_profile.loc[list(remaining), 'Recency'].idxmax()
remaining.remove(at_risk_cluster)

# Recent / Promising: lowest recency among the remaining clusters.
recent_cluster = cluster_profile.loc[list(remaining), 'Recency'].idxmin()
remaining.remove(recent_cluster)

# The final remaining cluster is labelled Regular Customers.
regular_cluster = remaining.pop()

segment_names = {
    high_value_cluster: 'High-Value Loyal',
    at_risk_cluster: 'At-Risk / Inactive',
    recent_cluster: 'Recent / Promising',
    regular_cluster: 'Regular Customers'
}

rfm['Segment'] = rfm['Cluster'].map(segment_names)

cluster_profile['Segment'] = cluster_profile.index.map(segment_names)
cluster_profile.sort_values('Monetary', ascending=False)

### Customer Segment Distribution

In [ ]:
segment_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(10, 6))
segment_counts.plot(kind='bar')
plt.title('Customer Segments')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig("images/customer_segments.png", dpi=150, bbox_inches="tight")
plt.show()

segment_counts.to_frame("Customers")

### Customer Segmentation Insights

- **High-Value Loyal:** Recent, frequent, and high-spending customers. Prioritise retention, VIP rewards, and personalised offers.
- **At-Risk / Inactive:** Customers with a long time since their last purchase and relatively low engagement. Use reactivation campaigns and targeted incentives.
- **Recent / Promising:** Customers who purchased recently but have not yet developed high frequency. Encourage repeat purchases with follow-up offers and recommendations.
- **Regular Customers:** Moderately engaged customers with repeat purchasing behaviour. Loyalty incentives and personalised promotions may increase their value.

## 11. Conclusions and Business Recommendations

In [ ]:
print("KEY PROJECT RESULTS")
print("-" * 50)
print(f"Cleaned transactions: {len(clean_df):,}")
print(f"Total revenue: £{total_revenue:,.2f}")
print(f"Unique orders: {total_orders:,}")
print(f"Identifiable customers: {total_customers:,}")
print(f"Top revenue country: {top_countries.index[0]}")
print(f"Top-selling product: {top_products.index[0]}")
print(f"Highest-revenue month: {monthly_sales.idxmax()}")

print("\nCustomer segment profile:")
display(cluster_profile.sort_values('Monetary', ascending=False))

### Business Recommendations

1. **Retain high-value loyal customers** with loyalty programmes, VIP treatment, and personalised offers.
2. **Re-engage inactive customers** through targeted email campaigns, time-limited discounts, and relevant product recommendations.
3. **Convert promising recent customers into repeat buyers** with follow-up offers after their initial purchases.
4. **Maintain stock availability for best-selling products** to reduce lost sales during periods of high demand.
5. **Protect the UK core market while developing strong international markets** such as the Netherlands, EIRE, Germany, and France.
6. **Prepare inventory and marketing before peak sales periods**, particularly the strong later months of the year.
7. **Repeat RFM segmentation periodically** so marketing strategies adapt as customer behaviour changes.

## 12. Export Project Outputs

In [ ]:
clean_df.to_csv('online_retail_cleaned.csv', index=False)
rfm.reset_index().to_csv('customer_segments.csv', index=False)
cluster_profile.reset_index().to_csv('cluster_profile.csv', index=False)

print("Files saved successfully:")
print("- online_retail_cleaned.csv")
print("- customer_segments.csv")
print("- cluster_profile.csv")
print("- images/monthly_revenue.png")
print("- images/top_products.png")
print("- images/top_countries.png")
print("- images/top_customers.png")
print("- images/elbow_method.png")
print("- images/silhouette_scores.png")
print("- images/customer_segments.png")

## Project Complete

The notebook has now completed:

**Data Understanding → Data Cleaning → Feature Engineering → Exploratory Analysis → Visualisation → RFM Analysis → K-Means Segmentation → Business Recommendations → Exported Results**